In [2]:
import numpy as np

In [172]:
def ReLU(x):
    return np.maximum(0, x)

def add_padding(x, padding):
    return np.pad(
        x,
        pad_width=((0, 0), (padding, padding), (padding, padding), (0, 0)),
        mode='constant',
        constant_values=0
    )

def conv2D(input_tensor, kernel, stride, padding):
    batch_size      = input_tensor.shape[0]
    height          = input_tensor.shape[1]
    width           = input_tensor.shape[2]
    output_channels = kernel.shape[3]
    kernel_size     = kernel.shape[0]

    output_h = (height - kernel_size + 2 * padding) // stride + 1
    output_w = (width  - kernel_size + 2 * padding) // stride + 1

    output_tensor = np.zeros((batch_size, output_h, output_w, output_channels))
    input_tensor  = add_padding(input_tensor, padding)

    for b in range(batch_size):
        for c in range(output_channels):
            for i in range(output_h):
                for j in range(output_w):
                    patch = input_tensor[
                        b,
                        i*stride : i*stride + kernel_size,
                        j*stride : j*stride + kernel_size,
                        :
                    ]
                    output_tensor[b, i, j, c] = np.sum(patch * kernel[:, :, :, c])

    return output_tensor


def dense_net_block(input_data, num_layers, growth_rate, kernels, kernel_size=(3, 3)):
    k = kernel_size[0]
    padding = (k - 1) //2
    feature_maps = [input_data]
    for l in range(num_layers):
        x0 = np.concatenate(feature_maps, axis=-1)
        x1 = ReLU(x0)
        x2 = conv2D(x1, kernels[l], 1, padding)
        feature_maps.append(x2)
    return np.concatenate(feature_maps, axis=-1)


In [176]:
np.random.seed(42)
X = np.random.randn(1, 2, 3, 2)
kernels = [np.random.randn(3, 3, 2 + i*1, 1) * 0.01 for i in range(2)]
print(dense_net_block(X, 2, 1, kernels))

[[[[ 0.49671415 -0.1382643  -0.0308579  -0.01845547]
   [ 0.64768854  1.52302986 -0.0041634  -0.0161227 ]
   [-0.23415337 -0.23413696 -0.02678915  0.00295656]]

  [[ 1.57921282  0.76743473  0.00334109 -0.04043312]
   [-0.46947439  0.54256004 -0.04493715  0.00983633]
   [-0.46341769 -0.46572975 -0.03523526  0.02832019]]]]
